<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: The alert budget

**[The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/)** · Anomaly Detection: Find What Is Unusual · Module 7, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** choose the threshold from the number of alerts that the team can review each day, instead of a guess of how much fraud there is, and to compute the cost per week of the frauds you miss and of the false alerts you send.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | Precision at k and recall, from [Why accuracy lies here](https://learning.nextia-ai.com/courses/anomaly-detection/m07/why-accuracy-lies-here/); Isolation Forest and its contamination setting, from [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M07-L02-the-alert-budget/alert-budget-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny example: ten payments and a budget

A clinic can see a fixed number of patients a day, so it sees the most urgent first. A review team works the same way. Here are 10 payments with a score from a detector. Four of them are planted frauds.

The function `tiny_cost(budget)` reviews the `budget` highest scores. A missed planted payment costs its amount plus a 15-dollar chargeback fee. A false alert costs 3 dollars of a reviewer's time. These costs are the course's **assumptions**.

> **Predict.** Which budget costs the least: 3, 5, 8 or 10 alerts?

> **Check.** You should see a cost of 171 dollars at 3 alerts, 39 at 5, 12 at 8 and 18 at 10.

In [ ]:
tiny = pd.DataFrame({
    "payment": [f"P{i}" for i in range(1, 11)],
    "score":   [0.91, 0.88, 0.80, 0.77, 0.70, 0.64, 0.60, 0.52, 0.47, 0.41],
    "planted": [True, True, False, True, False, False, True, False, False, False],
    "amount":  [40, 25, 60, 120, 15, 30, 18, 45, 22, 70],
})

def tiny_cost(budget, fee=15, false_alert=3):
    """Review the `budget` highest scores; every planted payment that is not reviewed is missed."""
    reviewed = tiny.sort_values("score", ascending=False).head(budget)
    missed = tiny[tiny.planted & ~tiny.payment.isin(reviewed.payment)]
    false_alerts = int((~reviewed.planted).sum())
    return {"budget": budget, "threshold": reviewed.score.min(),
            "precision": round(reviewed.planted.mean(), 3), "recall": round(reviewed.planted.sum() / tiny.planted.sum(), 3),
            "missed": len(missed), "missed cost": int(missed.amount.sum() + fee * len(missed)),
            "false alerts": false_alerts, "false-alert cost": false_alert * false_alerts,
            "cost": int(missed.amount.sum() + fee * len(missed) + false_alert * false_alerts)}

pd.DataFrame([tiny_cost(b) for b in (3, 5, 8, 10)])

The threshold is the score of the last payment that the team reviews. The budget sets it, not the other way round. More alerts find more frauds, until only false alerts are left to add.

## 2. Isolation Forest on Larkfield's payments

The detector is Isolation Forest on the behaviour features, as in Module 4. It learns from the reference months (June to August 2025) and scores the evaluation months (September to November 2025).

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
X_ref = scaler.transform(behaviour.loc[reference.index])
X_eval = scaler.transform(behaviour.loc[evaluation.index])
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(X_ref)
score = -forest.score_samples(X_eval)             # higher = more unusual
days = evaluation.ts.dt.floor("D").nunique()
print(f"{len(evaluation):,} evaluation payments over {days} days")

> **Check.** You should see 169,803 evaluation payments over 91 days.

## 3. The contamination guess

`contamination` tells Isolation Forest which share of the payments to call anomalies. It is a guess about the data, not about the team. Run the next cell for the default (`"auto"`) and for 1%.

In [ ]:
flag_auto = forest.predict(X_eval) == -1                   # contamination="auto", the default
cut_1pct = np.quantile(-forest.score_samples(X_ref), 0.99)   # contamination=0.01: the top 1% of the reference months
flag_1pct = score > cut_1pct
for name, flag in [("auto", flag_auto), ("0.01", flag_1pct)]:
    per_day = pd.Series(flag).groupby(evaluation.ts.dt.floor("D").to_numpy()).sum()
    print(f"contamination {name:>4}: {flag.sum() / days:6.1f} alerts a day on average (lowest {per_day.min()}, highest {per_day.max()})")

> **Check.** You should see about 386.3 alerts a day for `"auto"` and 25.7 for 0.01.

Neither number comes from the team. Sofia's team can review 50 a day: `"auto"` sends almost eight times more, and 0.01 leaves half of the team's time unused.

## 4. The threshold that the budget sets

`compare.daily_alerts(..., budget=50)` keeps the 50 highest scores of each day. The score of the 50th payment is that day's threshold.

In [ ]:
alerts = compare.daily_alerts(evaluation, score, budget=50)
threshold = alerts[alerts.reviewed].groupby("day").score.min()      # the score of the 50th payment of each day
print(threshold.describe()[["min", "50%", "max"]].round(3).to_string())
print("Black Friday 2025-11-28:", round(threshold["2025-11-28"], 3))

> **Check.** The threshold moves from 0.582 to 0.771; its median is 0.653.

Now use one fixed threshold, the median, on every day:

In [ ]:
fixed = threshold.median()                         # one fixed threshold for every day
per_day = (pd.Series(score >= fixed).groupby(evaluation.ts.dt.floor("D").to_numpy()).sum())
print(f"fixed threshold {fixed:.4f}: lowest {per_day.min()}, median {per_day.median():.0f}, highest {per_day.max()} alerts a day")
print("days with more than 50 alerts:", int((per_day > 50).sum()), "of", days)

> **Check.** You should see from 12 to 133 alerts a day, and 42 days with more than 50.

A fixed threshold sends a different number of alerts each day. The budget sends 50 every day, and the threshold moves instead.

## 5. The cost per week

Run the next cell. It computes the cost per week by hand, and compares it with `compare.score_payments`.

In [ ]:
row = compare.score_payments(evaluation, score, "Isolation Forest", "top 50 a day")
a = alerts.assign(planted=alerts.payment_id.isin(compare.planted_payments().payment_id))
missed = a[a.planted & ~a.reviewed]
false_alerts = int((a.reviewed & ~a.planted).sum())
weeks = days / 7
by_hand = {"missed payments a week": round(len(missed) / weeks, 1),
           "their amounts a week": round(missed.amount.sum() / weeks),
           "chargeback fees a week": round(15 * len(missed) / weeks),
           "false alerts a week": round(false_alerts / weeks, 1),
           "false-alert cost a week": round(3 * false_alerts / weeks),
           "cost per week": round((missed.amount.sum() + 15 * len(missed) + 3 * false_alerts) / weeks)}
print(by_hand)
print("compare.score_payments says:", row["cost per week ($)"])

> **Check.** By hand and from `compare.score_payments`, the cost per week is 3218 dollars.

Most of the cost is the money of the missed frauds (2,069 dollars a week). The false alerts cost 525 dollars a week.

## 6. The cost curve

> **Predict.** Will 100 alerts a day cost more or less than 50?

In [ ]:
curve = pd.DataFrame([compare.score_payments(evaluation, score, "Isolation Forest", budget=b)
                      for b in (10, 20, 30, 40, 50, 60, 80, 100, 150, 200)])
curve.insert(0, "budget", [10, 20, 30, 40, 50, 60, 80, 100, 150, 200])
cheapest = curve.loc[curve["cost per week ($)"].idxmin()]
fig, ax = plt.subplots()
ax.plot(curve.budget, curve["cost per week ($)"], marker="o")
ax.axvline(50, color="grey", ls=":", label="the team's 50 a day")
ax.set_xlabel("alerts a day (the budget)"); ax.set_ylabel("cost per week ($)"); ax.legend()
plt.show()
print("lowest cost:", int(cheapest["cost per week ($)"]), "dollars a week at", int(cheapest.budget), "alerts a day")
curve[["budget", "precision at budget", "recall", "false alerts per week", "cost per week ($)"]]

> **Check.** You should see the lowest cost, 2,802 dollars a week, at 80 alerts a day.

## 7. Check yourself

**Your turn 1.** What is the cost per week of Isolation Forest at 30 alerts a day? Write it as `cost_at_30`.

In [ ]:
cost_at_30 = compare.score_payments(evaluation, score, "Isolation Forest", budget=30)["cost per week ($)"]

In [ ]:
expect('cost_at_30', cost_at_30, 4780)

**Your turn 2.** The cost of a false alert is an assumption. Suppose a false alert costs 6 dollars instead of 3. Add a column `cost at 6 dollars` to `curve`, and write the budget with the lowest cost as `best_budget_at_6`.

Hint: `compare.score_payments` already counts 3 dollars per false alert, so add 3 more for each false alert per week.

In [ ]:
# 3 more dollars for each false alert: add 3 x (false alerts per week) to each cost
curve["cost at 6 dollars"] = curve["cost per week ($)"] + 3 * curve["false alerts per week"]
best_budget_at_6 = int(curve.loc[curve["cost at 6 dollars"].idxmin(), "budget"])

In [ ]:
expect_hash('best_budget_at_6', best_budget_at_6, '1a6562590ef19d10')

## Recap

- Choose the threshold from the budget: each day, the team reviews the top 50 scores, and the threshold moves.
- A contamination guess or a fixed threshold sends a number of alerts that nobody chose.
- Cost per week = missed planted payments (amount + 15 dollars) + 3 dollars per false alert. Its lowest point depends on these assumptions.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Evaluate without a full answer key](https://learning.nextia-ai.com/courses/anomaly-detection/m07/evaluate-without-a-full-answer-key/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.